In [0]:
import json
import re
import shutil
from datetime import datetime, timezone
from pathlib import Path
from pyspark.sql import functions as F
import sys
import importlib
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType


repo_root = "/Volumes/workspace/retail_dev_schema/retail_dev_dbx_vol"

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from utils import source_schemas as schema

In [0]:

ENTITIES = {
    "users": {"filename": "users.jsonl"},
    "products": {"filename": "products.jsonl"},
    "carts": {"filename": "carts.jsonl"}
}

BASE_PATH = "/Volumes/workspace/retail_dev_schema/retail_dev_dbx_vol"
LATEST_BATCH_ID = "batch_003"

In [0]:
FILENAME_USERS = ENTITIES.get('users')['filename']
LANDING_FILE_USERS = Path(BASE_PATH) / "landing" / LATEST_BATCH_ID / FILENAME_USERS
df_users_raw = (
    spark.read
    .format("json")
    .schema(schema.source_schema_users)
    .option("multiLine", "false")
    .option("mode", "FAILFAST")
    .load(str(LANDING_FILE_USERS))
)

FILENAME_CARTS = ENTITIES.get('carts')['filename']
LANDING_FILE_CARTS = Path(BASE_PATH) / "landing" / LATEST_BATCH_ID / FILENAME_CARTS
df_carts_raw = (
    spark.read
    .format("json")
    .schema(schema.source_schema_carts)
    .option("multiLine", "false")
    .option("mode", "FAILFAST")
    .load(str(LANDING_FILE_CARTS))
)

FILENAME_PRODUCTS = ENTITIES.get('products')['filename']
LANDING_FILE_PRODUCTS  = Path(BASE_PATH) / "landing" / LATEST_BATCH_ID / FILENAME_PRODUCTS
df_products_raw = (
    spark.read
    .format("json")
    .schema(schema.source_schema_products)
    .option("multiLine", "false")
    .option("mode", "FAILFAST")
    .load(str(LANDING_FILE_PRODUCTS))
)
# display(df_users_raw)
# display(df_carts_raw)
# display(df_products_raw)

In [0]:
df_users_raw_1 = df_users_raw.withColumn('country', df_users_raw.address.country) \
                             .withColumnRenamed("firstName", "first_name") \
                             .withColumnRenamed("lastName", "last_name") \
                             .withColumn("batch_id", F.lit(LATEST_BATCH_ID)) \
                             .withColumn("ingested_at", F.timestamp_seconds(F.unix_timestamp())) \
                             .drop("address")


df_products_raw_1 = df_products_raw.withColumnRenamed('discountPercentage', 'discount_percentage') \
                             .withColumn("batch_id", F.lit(LATEST_BATCH_ID)) \
                             .withColumn("ingested_at", F.timestamp_seconds(F.unix_timestamp()))


df_carts_raw1 = df_carts_raw.withColumnRenamed('userId', 'user_id') \
                             .withColumnRenamed("discountedTotal", "discounted_total") \
                             .withColumnRenamed("totalProducts", "total_products") \
                             .withColumnRenamed("totalQuantity", "total_quantity") \
                             .withColumn("batch_id", F.lit(LATEST_BATCH_ID)) \
                             .withColumn("ingested_at", F.timestamp_seconds(F.unix_timestamp()))


display(df_products_raw_1)

In [0]:
target_table_users = "workspace.retail_dev_schema.bronze_users"

(
    df_users_raw_1.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{LATEST_BATCH_ID}'")
    .saveAsTable(target_table_users)
)


In [0]:
target_table_products = "workspace.retail_dev_schema.bronze_products"

(
    df_products_raw_1.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{LATEST_BATCH_ID}'")
    .saveAsTable(target_table_products)
)

In [0]:

target_table_carts = "workspace.retail_dev_schema.bronze_carts"

(
    df_carts_raw1.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"batch_id = '{LATEST_BATCH_ID}'")
    .saveAsTable(target_table_carts)
) 